# VGG-16 — Repeated 3 x 3 Convolutions and PyTorch Implementation

## Overview

This notebook implements VGG-16 from first principles in PyTorch. The central architectural idea is building depth from repeated small convolutions followed by regular spatial downsampling. The implementation uses Imagenette with 10 ImageNet-derived classes, and the notebook demonstrates the data pipeline, model construction, inspection, training, evaluation, and prediction workflow.


## Table of Contents

- [Overview](#overview)
- [Learning Objectives](#learning-objectives)
- [Architecture Theory](#architecture-theory)
- [Environment Setup](#environment-setup)
- [Configuration](#configuration)
- [Reproducibility](#reproducibility)
- [Device Selection](#device-selection)
- [Dataset Download](#dataset-download)
- [Data Augmentation and Preprocessing](#data-augmentation-and-preprocessing)
- [Dataset Construction and Splitting](#dataset-construction-and-splitting)
- [Data Loaders](#data-loaders)
- [Dataset Inspection](#dataset-inspection)
- [Architecture Components](#architecture-components)
- [Complete Model](#complete-model)
- [Model Initialization](#model-initialization)
- [Model Inspection](#model-inspection)
- [Parameter Count](#parameter-count)
- [Training Objective and Optimizer](#training-objective-and-optimizer)
- [Training Function](#training-function)
- [Evaluation Function](#evaluation-function)
- [Training Loop](#training-loop)
- [Test Evaluation](#test-evaluation)
- [Key Takeaways](#key-takeaways)


## Learning Objectives

By the end of this notebook, we should understand:

- why repeated `3 x 3` convolutions can replace larger receptive fields.
- how VGG expands channels while reducing spatial resolution.
- where the large fully connected classifier concentrates many parameters.
- how the Imagenette training split is divided for validation.
- how to inspect feature-map shapes and parameter counts.


## Architecture Theory

VGG-16 makes the network design deliberately uniform. Instead of mixing many kernel sizes, it stacks `3 x 3` convolutions, uses max pooling between stages, and doubles the channel count as spatial resolution decreases. Two stacked `3 x 3` convolutions have the receptive field of a `5 x 5` convolution with additional nonlinearities; three stacked `3 x 3` convolutions approximate a `7 x 7` receptive field.

The implementation follows Configuration D: five convolutional stages and a large classifier. Most storage cost is concentrated in the fully connected layers after the `512 x 7 x 7` feature tensor is flattened.

| Stage | Operation | Output Shape |
| --- | --- | --- |
| Input | RGB image | `(N, 3, 224, 224)` |
| Block 1 | `3 x 3` conv x2, max pool | `(N, 64, 112, 112)` |
| Block 2 | `3 x 3` conv x2, max pool | `(N, 128, 56, 56)` |
| Block 3 | `3 x 3` conv x3, max pool | `(N, 256, 28, 28)` |
| Block 4 | `3 x 3` conv x3, max pool | `(N, 512, 14, 14)` |
| Block 5 | `3 x 3` conv x3, max pool | `(N, 512, 7, 7)` |
| Classifier | `25088 -> 4096 -> 4096 -> classes` | `(N, 10)` |


## Environment Setup

The notebook uses PyTorch for tensor computation and neural network modules, torchvision for datasets and transforms, NumPy and Python randomness for reproducibility, and Matplotlib for visualization.


In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.datasets.utils import download_and_extract_archive


## Configuration

The following constants define the experiment. They are kept from the source script so the notebook remains traceable to the original implementation.


In [ ]:
DATA_DIRECTORY = "./data"

IMAGENETTE_URL = (
    "https://s3.amazonaws.com/"
    "fast-ai-imageclas/"
    "imagenette2-320.tgz"
)

IMAGENETTE_FOLDER = "imagenette2-320"

IMAGE_SIZE = 224
BATCH_SIZE = 16
LEARNING_RATE = 0.01
NUM_EPOCHS = 30

VALIDATION_RATIO = 0.2
RANDOM_SEED = 42
NUMBER_OF_WORKERS = 0

NUMBER_OF_CLASSES = 10

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


## Reproducibility

Random seeds make dataset splitting, parameter initialization, and stochastic operations easier to reproduce across runs. Exact determinism can still depend on hardware kernels and backend behavior.


In [ ]:
def set_seed(seed: int) -> None:
    """Set random seed for reproducibility."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## Device Selection

The model is moved to the best available device. CUDA is preferred when available, followed by Apple MPS, and then CPU.


In [ ]:
def get_device() -> torch.device:
    """Return the best available device (GPU if available, else CPU)."""

    if torch.cuda.is_available():
        return torch.device("cuda")

    if torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")

device = get_device()

print(f"Using device: {device}")


## Dataset Download

Imagenette is a ten-class subset of ImageNet. The helper below downloads and prepares the archive if it is not already present.


In [ ]:
def prepare_imagenette(data_directory: str) -> Path:
    """Download and prepare the Imagenette dataset."""

    root_directoty = Path(data_directory)

    dataset_directory = root_directoty / IMAGENETTE_FOLDER

    train_directory = dataset_directory / "train"
    validation_directory = dataset_directory / "val"

    if (train_directory.exists() and validation_directory.exists()):
        print(f"Imagenette dataset already exists at {dataset_directory}.")
        return dataset_directory

    root_directoty.mkdir(parents=True, exist_ok=True)

    print(f"Downloading Imagenette dataset from {IMAGENETTE_URL}...")

    download_and_extract_archive(
        url=IMAGENETTE_URL,
        download_root=root_directoty,
        extract_root=root_directoty,
    )

    return dataset_directory

imagenette_directory = prepare_imagenette(DATA_DIRECTORY)


## Data Augmentation and Preprocessing

Training transforms are stochastic and regularize the model through random crops or flips. Evaluation transforms are deterministic so validation and test metrics are comparable across runs. Normalization maps image channels into the scale expected by the training recipe.


In [ ]:
training_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=(IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.8, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

evaluation_transform = transforms.Compose([
    transforms.Resize(size=256),
    transforms.CenterCrop(size=IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


## Dataset Construction and Splitting

The dataset objects define the supervised image-classification task and its train, validation, and test usage. The source implementation creates validation data from the training split and reserves the official evaluation split for final testing where applicable.


In [ ]:
train_directory = imagenette_directory / "train"
test_directory = imagenette_directory / "val"

# ------------------------------------------------------------
# Training view
# ------------------------------------------------------------

training_dataset_full = datasets.ImageFolder(
    root=train_directory,
    transform=training_transform
)

# ------------------------------------------------------------
# Validation view
#
# This points to the same images as training_dataset_full,
# but uses deterministic evaluation transforms.
# ------------------------------------------------------------

validation_dataset_full = datasets.ImageFolder(
    root=train_directory,
    transform=evaluation_transform
)

# ------------------------------------------------------------
# Official Imagenette validation split
# is used here as the final test set.
# ------------------------------------------------------------

test_dataset = datasets.ImageFolder(
    root=test_directory,
    transform=evaluation_transform
)

# ------------------------------------------------------------
# Reproducible train / validation split
# ------------------------------------------------------------

number_of_samples = len(training_dataset_full)
number_of_validation_samples = int(VALIDATION_RATIO * number_of_samples)

generator = torch.Generator().manual_seed(RANDOM_SEED)

indices = torch.randperm(number_of_samples, generator=generator).tolist()

validation_indices = indices[:number_of_validation_samples]
training_indices = indices[number_of_validation_samples:]

training_dataset = Subset(training_dataset_full, training_indices)
validation_dataset = Subset(validation_dataset_full, validation_indices)

print(
    f"Training samples:   "
    f"{len(training_dataset):,}"
)

print(
    f"Validation samples: "
    f"{len(validation_dataset):,}"
)

print(
    f"Test samples:       "
    f"{len(test_dataset):,}"
)

print(
    f"Number of classes:  "
    f"{len(training_dataset_full.classes)}"
)


## Data Loaders

Data loaders batch examples, optionally shuffle the training subset, and pin memory when CUDA is used. Validation and test loaders are not shuffled because their order does not affect the metrics.


In [ ]:
pin_memory = device.type == "cuda"

training_loader = DataLoader(
    dataset=training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=pin_memory
)

validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=pin_memory
)


## Dataset Inspection

A small batch visualization helps verify that augmentation, normalization, and labels are wired correctly. The display reverses normalization before plotting.


In [ ]:
def show_training_examples(data_loader: DataLoader, number_of_images: int = 8) -> None:
    images, labels = next(iter(data_loader))
    images = images[:number_of_images]
    labels = labels[:number_of_images]

    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    images = (images.cpu() * std + mean).clamp(0, 1)

    class_names = training_dataset_full.classes
    fig, axes = plt.subplots(1, number_of_images, figsize=(14, 2.5))

    for axis, image, label in zip(axes, images, labels):
        axis.imshow(image.permute(1, 2, 0))
        axis.set_title(class_names[int(label)], fontsize=8)
        axis.axis("off")

    plt.tight_layout()
    plt.show()

show_training_examples(training_loader)


## Architecture Components

### VGG Convolutional Block

A VGG block repeats `3 x 3` convolutions at a fixed channel width, then downsamples with max pooling. Repetition increases effective receptive field while adding nonlinearities.


In [ ]:
class VGGBlock(nn.Module):
    """Repeated VGG convolutional block followed by max pooling."""
    def __init__(self, in_channels: int, out_channels: int, number_of_convolutions: int) -> None:
        super().__init__()

        layers: list[nn.Module] = []

        current_channels = in_channels

        for _ in range(number_of_convolutions):
            layers.append(
                nn.Conv2d(
                    in_channels=current_channels,
                    out_channels=out_channels,
                    kernel_size=3,
                    stride=1,
                    padding=1
                )
            )

            layers.append(nn.ReLU(inplace=True))

            current_channels = out_channels

        self.convolution_layers = nn.Sequential(*layers)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.convolution_layers(x)
        x = self.pool(x)
        return x


## Architecture Components

### VGG Classifier

The classifier flattens the final `512 x 7 x 7` tensor and applies large fully connected layers with dropout.


In [ ]:
class VGGClassifier(nn.Module):
    """Fully connected VGG classifier head."""
    def __init__(self, number_of_classes: int) -> None:
        super().__init__()

        self.flatten = nn.Flatten(start_dim=1)

        self.fc1 = nn.Linear(in_features=512 * 7 * 7, out_features=4096)
        self.relu1 = nn.ReLU(inplace=True)
        self.dropout1 = nn.Dropout(0.5)

        self.fc2 = nn.Linear(in_features=4096, out_features=4096)
        self.relu2 = nn.ReLU(inplace=True)
        self.dropout2 = nn.Dropout(0.5)

        self.output_layer = nn.Linear(in_features=4096, out_features=number_of_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        logits = self.output_layer(x)

        return logits


## Complete Model

The complete VGG-16 model stacks five convolutional blocks before the classifier. The implementation keeps the model modular by separating feature blocks from the classifier head.


In [ ]:
class VGG16(nn.Module):
    """VGG-16 Configuration D image classification network."""
    def __init__(self, number_of_classes: int = NUMBER_OF_CLASSES) -> None:
        super().__init__()

        self.block1 = VGGBlock(in_channels=3, out_channels=64, number_of_convolutions=2)

        self.block2 = VGGBlock(in_channels=64, out_channels=128, number_of_convolutions=2)

        self.block3 = VGGBlock(in_channels=128, out_channels=256, number_of_convolutions=3)

        self.block4 = VGGBlock(in_channels=256, out_channels=512, number_of_convolutions=3)

        self.block5 = VGGBlock(in_channels=512, out_channels=512, number_of_convolutions=3)

        self.adaptive_pool = nn.AdaptiveAvgPool2d(output_size=(7, 7))

        self.classifier = VGGClassifier(number_of_classes=number_of_classes)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)

        x = self.block2(x)

        x = self.block3(x)

        x = self.block4(x)

        x = self.block5(x)

        x = self.adaptive_pool(x)

        logits = self.classifier(x)

        return logits


## Model Initialization

The model is instantiated with the configured number of classes and moved to the selected device so parameters and input tensors live on the same backend.


In [ ]:
model = VGG16(number_of_classes=NUMBER_OF_CLASSES).to(device)

print("Model architecture:")
print(model)


## Model Inspection

Shape inspection is a lightweight sanity check. It verifies that a synthetic input flows through the model and produces logits with the expected class dimension.


In [ ]:
def inspect_model_shapes(model: VGG16, device: torch.device) -> None:
    """Inspect the shapes of the model's layers."""

    sample_batch = torch.randn(
        2,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        device=device,
    )

    model.eval()

    with torch.inference_mode():
        print("Shape inspection:")

        print(f"Input shape: {tuple(sample_batch.shape)}")

        x = model.block1(sample_batch)
        print(f"After Block 1: {tuple(x.shape)}")

        x = model.block2(x)
        print(f"After Block 2: {tuple(x.shape)}")

        x = model.block3(x)
        print(f"After Block 3: {tuple(x.shape)}")

        x = model.block4(x)
        print(f"After Block 4: {tuple(x.shape)}")

        x = model.block5(x)
        print(f"After Block 5: {tuple(x.shape)}")

        x = model.adaptive_pool(x)
        print(f"After Adaptive Pool: {tuple(x.shape)}")

        x = model.classifier.flatten(x)
        print(f"After Flatten: {tuple(x.shape)}")

        x = model.classifier.fc1(x)
        print(f"After FC1: {tuple(x.shape)}")

        x = model.classifier.relu1(x)
        print(f"After ReLU1: {tuple(x.shape)}")

        x = model.classifier.dropout1(x)
        print(f"After Dropout1: {tuple(x.shape)}")

        x = model.classifier.fc2(x)
        print(f"After FC2: {tuple(x.shape)}")

        x = model.classifier.relu2(x)
        print(f"After ReLU2: {tuple(x.shape)}")

        x = model.classifier.dropout2(x)
        print(f"After Dropout2: {tuple(x.shape)}")

        logits = model.classifier.output_layer(x)
        print(f"After Output Layer: {tuple(logits.shape)}")

inspect_model_shapes(model=model, device=device)


## Parameter Count

Parameter count is a rough measure of model capacity and storage cost. It does not fully measure computation, but it helps compare architectural trade-offs.


In [ ]:
def count_parameters(model: nn.Module) -> tuple[int, int]:
    """Count the number of trainable and non-trainable parameters in the model."""

    total_parameters = sum(p.numel() for p in model.parameters())
    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return total_parameters, trainable_parameters

total_params, trainable_params = count_parameters(model)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


## Training Objective and Optimizer

For multi-class classification, the network outputs logits. `CrossEntropyLoss` combines log-softmax and negative log-likelihood, so the model should not apply softmax before the loss. The optimizer updates trainable parameters according to the configured learning rate.


In [ ]:
# CrossEntropyLoss expects:
#
# logits:
#     shape = (N, 10)
#
# labels:
#     shape = (N,)
#     dtype = torch.int64
#
# Do not apply Softmax before CrossEntropyLoss.

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)


## Training Function

One training epoch performs forward propagation, loss computation, gradient reset, backpropagation, optimizer update, and metric accumulation.


In [ ]:
def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device
) -> tuple[float, float]:
    """Train the model for one epoch."""

    model.train()

    accumulated_loss = 0.0

    number_of_correct_predictions = 0

    number_of_samples = 0

    for images, labels in data_loader:
        images = images.to(device, non_blocking=(device.type == "cuda"))
        labels = labels.to(device, non_blocking=(device.type == "cuda"))

        optimizer.zero_grad(set_to_none=True)

        # ----------------------------------------------------
        # Forward propagation
        #
        # images:
        #     (N, 3, 224, 224)
        #
        # logits:
        #     (N, 10)
        # ----------------------------------------------------
        logits = model(images)

        loss = loss_function(logits, labels)

        loss.backward()

        optimizer.step()

        accumulated_loss += loss.item() * images.size(0)

        predictions = torch.argmax(logits, dim=1)

        number_of_correct_predictions += (predictions == labels).sum().item()

        number_of_samples += images.size(0)

    average_loss = accumulated_loss / number_of_samples
    accuracy = number_of_correct_predictions / number_of_samples

    return average_loss, accuracy


## Evaluation Function

Evaluation disables gradient tracking and measures loss and accuracy without updating parameters.


In [ ]:
def evaluate(
        model: nn.Module,
        data_loader: DataLoader,
        loss_function: nn.Module,
        device: torch.device
) -> tuple[float, float]:
    """Evaluate the model on a validation or test set."""

    model.eval()

    accumulated_loss = 0.0

    number_of_correct_predictions = 0

    number_of_samples = 0

    with torch.inference_mode():
        for images, labels in data_loader:
            images = images.to(device, non_blocking=(device.type == "cuda"))
            labels = labels.to(device, non_blocking=(device.type == "cuda"))

            # ----------------------------------------------------
            # Forward propagation
            #
            # images:
            #     (N, 3, 224, 224)
            #
            # logits:
            #     (N, 10)
            # ----------------------------------------------------
            logits = model(images)

            loss = loss_function(logits, labels)

            accumulated_loss += loss.item() * images.size(0)

            predictions = torch.argmax(logits, dim=1)

            number_of_correct_predictions += (predictions == labels).sum().item()

            number_of_samples += images.size(0)

    average_loss = accumulated_loss / number_of_samples
    accuracy = number_of_correct_predictions / number_of_samples

    return average_loss, accuracy


## Training Loop

The training loop coordinates epochs, validation, and history recording. Running the next cell can take a long time because it executes the full configured experiment.


In [ ]:
training_loss_history: list[float] = []

validation_loss_history: list[float] = []

training_accuracy_history: list[float] = []

validation_accuracy_history: list[float] = []

for epoch in range(1, NUM_EPOCHS + 1):
    training_loss, training_accuracy = train_one_epoch(
        model=model,
        data_loader=training_loader,
        loss_function=loss_function,
        optimizer=optimizer,
        device=device
    )

    validation_loss, validation_accuracy = evaluate(
        model=model,
        data_loader=validation_loader,
        loss_function=loss_function,
        device=device
    )

    training_loss_history.append(training_loss)

    validation_loss_history.append(validation_loss)

    training_accuracy_history.append(training_accuracy)

    validation_accuracy_history.append(validation_accuracy)

    print(
        f"Epoch "
        f"[{epoch:02d}/{NUM_EPOCHS:02d}] | "
        f"Training loss: "
        f"{training_loss:.4f} | "
        f"Training accuracy: "
        f"{training_accuracy:.2%} | "
        f"Validation loss: "
        f"{validation_loss:.4f} | "
        f"Validation accuracy: "
        f"{validation_accuracy:.2%}"
    )


## Test Evaluation

The test set is used after training for final evaluation. Training data optimizes parameters, validation data monitors model selection, and test data estimates final generalization.


In [ ]:
test_loss, test_accuracy = evaluate(
    model=model,
    data_loader=test_loader,
    loss_function=loss_function,
    device=device
)

print(
    f"Test loss: "
    f"{test_loss:.4f} | "
    f"Test accuracy: "
    f"{test_accuracy:.2%}"
)


## Key Takeaways

- VGG-16 shows how a simple repeated block can define a deep CNN.
- Spatial resolution is halved stage by stage while channel depth increases.
- The classifier dominates the parameter count compared with the convolutional blocks.
